# Compound AI Systems Exercise — Assessment 01

**Role context:** Researcher interview, live notebook prototyping  
**Time limit: 45 minutes. No AI coding assistance.**

Welcome. In this session you will design and prototype a small **compound AI system** — a pipeline that chains multiple model calls to solve a realistic business problem. You may use plain Python and the provided `call_llm` helper. Specialized orchestration frameworks are not required.

Focus on: how you decompose the problem, how intermediate outputs flow between calls, and how you validate the final result.


## Problem Statement

Acme Corp's Travel & Expense policy has grown organically. Legal and Finance suspect the document contains **internal conflicts and ambiguous exceptions** that employees exploit (or that cause legitimate reimbursements to be rejected inconsistently).

Your task: build a system that, given a multi-section policy document, produces a structured report of **policy conflicts, ambiguities, and risky exception pathways**.

This is an enterprise document-analysis problem. The output should help a compliance analyst prioritize what to fix — not merely summarize the policy.


## Functional Requirements

**Input**
- `policy_text: str` — full policy document (plain text, sectioned)

**Output** — a Python `dict` (JSON-serializable) with:
```python
{
  "conflicts": [
    {
      "id": "C1",
      "summary": "...",
      "sections_involved": ["1.1", "4.2"],
      "severity": "high" | "medium" | "low",
      "explanation": "why these clauses conflict",
      "example_scenario": "concrete employee action that exposes the conflict"
    }
  ],
  "ambiguities": [
    {
      "id": "A1",
      "summary": "...",
      "sections_involved": ["..."],
      "explanation": "...",
      "clarifying_question": "question Legal should answer"
    }
  ],
  "exception_risks": [
    {
      "id": "R1",
      "summary": "...",
      "sections_involved": ["..."],
      "abuse_potential": "high" | "medium" | "low",
      "explanation": "..."
    }
  ],
  "analyst_summary": "2-4 sentence executive brief for Legal/Finance"
}
```

**Constraints**
- Rely only on the provided policy text (no external knowledge of Acme).
- Prefer precise section references (e.g. `"4.2"`) over vague citations.
- If the model is uncertain, still surface the issue with lower severity rather than dropping it.
- Handle empty / very short input gracefully (return empty lists + a summary noting insufficient content).

**Edge cases**
- Policies with no real conflicts (return empty `conflicts`, still check ambiguities).
- Overlapping exception clauses that create loopholes.
- Soft language ("may", "should") that creates ambiguity without hard contradiction.

**Desired behavior**
A compliance analyst should be able to act on your report without re-reading the entire policy.


## Available Tools & Starter Code

Configure API access, then run the cells below. See comments in the setup cell for environment variables.


In [ ]:
import json
import os
import re
from typing import Any, Optional

# ---------------------------------------------------------------------------
# LLM configuration
# ---------------------------------------------------------------------------
# Set one of the following before running (or export in your shell):
#
#   export OPENAI_API_KEY="sk-..."
#   # optional: export OPENAI_MODEL="gpt-4o-mini"
#
#   export ANTHROPIC_API_KEY="sk-ant-..."
#   # optional: export ANTHROPIC_MODEL="claude-3-5-haiku-latest"
#
# If neither key is set, call_llm runs in MOCK mode so the notebook still
# executes for smoke-testing. Mock responses are placeholders — they are not
# a substitute for a real model during practice.
# ---------------------------------------------------------------------------

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-3-5-haiku-latest")

USE_MOCK = not (OPENAI_API_KEY or ANTHROPIC_API_KEY)
if USE_MOCK:
    print("⚠️  No API key found. call_llm is in MOCK mode.")
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"✓ Using {provider} API")


def _mock_response(prompt: str, system_prompt: str = "") -> str:
    """Lightweight deterministic mock for offline smoke tests."""
    combined = f"{system_prompt}\n{prompt}".lower()
    # Return minimal JSON-ish or text so downstream parsing can be exercised.
    if "json" in combined:
        return json.dumps({"mock": True, "note": "Replace with real API key for useful output", "items": []})
    return "[MOCK RESPONSE] Configure OPENAI_API_KEY or ANTHROPIC_API_KEY for real model output."


def call_llm(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
    temperature: float = 0.2,
    max_tokens: int = 2000,
) -> str:
    """Call a chat LLM. Credentials come from environment variables only."""
    if USE_MOCK:
        return _mock_response(prompt, system_prompt)

    try:
        if OPENAI_API_KEY:
            from openai import OpenAI

            client = OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                temperature=temperature,
                max_tokens=max_tokens,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": prompt},
                ],
            )
            return (resp.choices[0].message.content or "").strip()

        from anthropic import Anthropic

        client = Anthropic(api_key=ANTHROPIC_API_KEY)
        resp = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", None) == "text"
        ).strip()
    except Exception as e:
        return f"[API ERROR] {type(e).__name__}: {e}"


def extract_json(text: str) -> Any:
    """Best-effort parse of a JSON object/array from model output."""
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    fence = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except json.JSONDecodeError:
            pass
    for opener, closer in (("{", "}"), ("[", "]")):
        start, end = text.find(opener), text.rfind(closer)
        if start != -1 and end > start:
            try:
                return json.loads(text[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError(f"Could not parse JSON from model output:\n{text[:500]}")


# Smoke test
print("call_llm ready.")
print(call_llm("Reply with exactly: OK", system_prompt="Be terse.")[:80])


In [ ]:
# Sample policy document for this exercise
POLICY_TEXT = """
CORPORATE TRAVEL & EXPENSE POLICY (v3.2) — Acme Corp

SECTION 1 — Booking Authority
1.1 Employees may book domestic flights under $800 without manager approval.
1.2 International flights require VP approval regardless of cost.
1.3 The Travel Desk may override booking rules for executive assistants supporting C-level staff.

SECTION 2 — Lodging
2.1 Hotel stays are capped at $250/night in Tier-1 cities and $175/night elsewhere.
2.2 Employees attending company-sponsored conferences may book the official conference hotel
    even if the nightly rate exceeds the standard cap, with no additional approval required.
2.3 Suite upgrades are prohibited unless medically necessary and approved by HR.

SECTION 3 — Meals & Incidentals
3.1 Daily meal allowance is $75 domestic / $100 international.
3.2 Client entertainment meals may exceed the daily allowance if a receipt and client names
    are attached; no hard dollar cap applies.
3.3 Alcohol is reimbursable only when accompanying a client meal.

SECTION 4 — Exceptions & Escalation
4.1 Any expense over $2,000 requires Finance Controller pre-approval.
4.2 The Travel Desk exception process (Form T-9) can approve any single expense up to $5,000
    without Finance Controller involvement if submitted within 48 hours of booking.
4.3 Retroactive exceptions after travel are not permitted.

SECTION 5 — Card & Reimbursement
5.1 All travel must be charged to the corporate card when available.
5.2 Personal card use is allowed only when the corporate card is declined; reimbursement
    then requires manager sign-off within 5 business days.
5.3 Employees in Sales Ops are exempt from the corporate-card requirement and may use
    personal cards for all travel expenses under $1,500.
"""

print(f"Policy length: {len(POLICY_TEXT)} characters")
print(POLICY_TEXT[:400], "...")


## Coding Section

Implement `analyze_policy_conflicts`. You decide how many model calls to make and how to structure them. Do not change the function signature or the required output schema.


In [ ]:
def analyze_policy_conflicts(policy_text: str) -> dict:
    """Analyze a policy document for conflicts, ambiguities, and exception risks.

    Args:
        policy_text: Full policy document as plain text.

    Returns:
        dict matching the schema in Functional Requirements.
    """
    # TODO: Implement your compound AI system here.
    # Use call_llm(...) for model calls. You may use extract_json(...) if helpful.
    raise NotImplementedError("Implement analyze_policy_conflicts")


## Example Input / Output

Using `POLICY_TEXT` above, a strong solution might surface findings such as:

- A **conflict** between the $2,000 Finance Controller rule (4.1) and the Travel Desk T-9 path allowing up to $5,000 without Finance (4.2).
- An **ambiguity** around whether conference hotels (2.2) still need other approvals when combined with international travel rules.
- An **exception risk** where Sales Ops personal-card exemption (5.3) interacts with meal/entertainment rules.

Illustrative (abbreviated) shape — **not** a gold answer:

```python
{
  "conflicts": [
    {
      "id": "C1",
      "summary": "Finance Controller pre-approval vs Travel Desk T-9 override",
      "sections_involved": ["4.1", "4.2"],
      "severity": "high",
      "explanation": "...",
      "example_scenario": "Employee books $4,200 itinerary via T-9 within 48h, bypassing Controller."
    }
  ],
  "ambiguities": [...],
  "exception_risks": [...],
  "analyst_summary": "..."
}
```

Your architecture is up to you; multiple valid designs exist.


## Run Your Solution

When ready, execute the cell below. Aim for a coherent report on `POLICY_TEXT` within the time box.


In [ ]:
# Uncomment when implemented:
# result = analyze_policy_conflicts(POLICY_TEXT)
# print(json.dumps(result, indent=2))


## Time Limit

**Time limit: 45 minutes. No AI coding assistance.**

You may consult standard Python documentation. Prefer a working end-to-end prototype over an incomplete elaborate design.
